In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "BNBUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-09-01 00:00:00+00:00,857.66,857.67,857.24,857.66,251.305,2025-09-01 00:00:59.999999+00:00,215467.75012,654,192.217,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000000,0.000000,0.000000,NaN,NaN
1,2025-09-01 00:01:00+00:00,857.67,858.16,857.67,858.15,140.110,2025-09-01 00:01:59.999999+00:00,120206.45623,490,82.628,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.010994,0.006108,0.004886,NaN,NaN
2,2025-09-01 00:02:00+00:00,858.16,858.16,857.55,857.75,207.449,2025-09-01 00:02:59.999999+00:00,177947.04945,566,66.245,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.001604,0.004262,-0.002658,NaN,NaN
3,2025-09-01 00:03:00+00:00,857.76,858.25,857.75,857.81,315.626,2025-09-01 00:03:59.999999+00:00,270770.38427,391,254.225,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.000540,0.002635,-0.003175,NaN,NaN
4,2025-09-01 00:04:00+00:00,857.80,857.81,856.12,856.13,415.090,2025-09-01 00:04:59.999999+00:00,355712.34813,1816,55.089,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.068544,-0.018539,-0.050005,NaN,NaN


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 284,601
[info] optuna train rows: 182,144
[info] valid rows:        45,536
[info] test rows:         56,921


In [9]:
study = optuna.create_study(direction="maximize")
objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-19 22:25:12,533] A new study created in memory with name: no-name-74944e34-13c2-4e6d-9cf9-50299bcd97b5


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:47<?, ?it/s]

Best trial: 0. Best value: -0.0101555:   0%|          | 0/50 [00:47<?, ?it/s]

Best trial: 0. Best value: -0.0101555:   2%|▏         | 1/50 [00:47<38:58, 47.73s/it]

...

Best trial: 45. Best value: 0.0145564:  98%|█████████▊| 49/50 [14:57<00:08,  8.97s/it]

Best trial: 45. Best value: 0.0145564:  98%|█████████▊| 49/50 [14:58<00:08,  8.97s/it]

Best trial: 45. Best value: 0.0145564: 100%|██████████| 50/50 [14:58<00:00, 18.53s/it]

Best trial: 45. Best value: 0.0145564: 100%|██████████| 50/50 [14:58<00:00, 17.96s/it]

[I 2026-03-19 22:40:10,534] Trial 49 finished with value: -0.004649794343219021 and parameters: {'n_estimators': 300, 'max_depth': 11, 'min_samples_split': 21, 'min_samples_leaf': 5, 'max_features': 1.0, 'bootstrap': False}. Best is trial 45 with value: 0.014556397039771966.

[optuna] best trial
value: 0.014556
params:
  n_estimators: 200
  max_depth: 5
  min_samples_split: 17
  min_samples_leaf: 3
  max_features: 0.5
  bootstrap: False


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final rf...


[training] done in 5.94s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:      0.402297
Test IC:       -0.000000
Train Rank IC: 0.015720
Test Rank IC:  nan
Train RMSE:    0.001995
Test RMSE:     0.001665


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
mom_x_imb           0.308706
dist_ma_15          0.163719
mom_10              0.105054
mom_60              0.103848
mom_15              0.074507
mom_30              0.050933
mom_5               0.047871
vol_30              0.034359
macd_hist           0.029767
dist_ma_30          0.024179
range_5             0.014087
mom_3               0.010686
atr_norm            0.009059
vol_5               0.007680
dist_ma_5           0.004818
range_15            0.003820
vol_regime_ratio    0.002934
vol_15              0.001900
volume_mom_5        0.001291
num_trades_mom_5    0.000782
vol_ratio_5_30      0.000000
dist_ma_15_z        0.000000
range_ratio         0.000000
trend_strength      0.000000
is_trending         0.000000
bar_range           0.000000
is_high_vol         0.000000
imbalance           0.000000
imbalance_15        0.000000
imbalance_5         0.000000
volume_z            0.000000
trades_z            0.000000
mr_x_vol            0.000000
trend_x_imb

In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/rf/BNBUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/rf/BNBUSDT__h5_model.joblib
[saved] features -> models/rf/BNBUSDT__h5_feature_cols.json
[saved] feature importance -> models/rf/BNBUSDT__h5_feature_importance.csv
[saved] metadata -> models/rf/BNBUSDT__h5_meta.json
